# U5-L3：LLM 对话

**课程目标**

1. 能说明大语言模型怎样利用训练所得和当前上下文生成回答，并指出流畅的回答仍可能有事实错误。
2. 能区分模型与角色提示词：换提示词可以影响回答的身份、语气、长度和范围，不能凭空增加实时信息或工具。
3. 能按身份、语气、知识范围、不能做什么编写提示词，用同一个问题比较不同设定的表现。
4. 能启动持续语音对话，用 `wait()` 等待会话，并在中断后请求取消、检查任务是否结束。


## 一、本课要解决的问题

前两课已经把语音转成文字，也让机器人说出了指定文字。要与来宾交流，还需要根据问题组织回答。

大语言模型可以生成回答。回答内容受模型训练所得、当前设定和对话内容影响；同一个问题多问几次，措辞可能不同，也可能出现事实错误。

本课用同一个问题比较不同角色提示词，观察回答的语气、长度和范围，并尝试把它配置成适合实训室或展厅的讲解员。


## 二、大语言模型在做什么

语音对话通常要经过语音识别、回答生成和语音合成。大语言模型主要参与其中的回答生成：根据输入的文字及上下文，生成后续内容。

`chat()` 把这些环节组织在一个持续会话中。本课通过接口使用它，重点观察角色设定怎样影响回答。


### 2.1 用“文字接龙”理解生成过程

可以把大语言模型生成回答的过程，初步想成“接着前文往下写”。例如“欢迎来到实训室，请先……”后面，可以接“阅读操作要求”，也可以接其他内容；选择与当前问题、前文和模型训练所得有关。

模型通常逐步生成文本，每次处理的单位叫 **token**，可能对应一个字、一个词或词的一部分。生成出的内容加入上下文后，再影响后面的内容。大语言模型的英文缩写是 **LLM**。

“文字接龙”只是帮助理解逐步生成的比喻。模型在训练中也学习了语言、知识及解决问题的模式，能够完成解释、写作和代码等任务；不能因此认定它只会检查句子是否通顺。它的回答可以很流畅，同时仍有事实错误。


### 2.2 它的知识从哪来

模型在训练时从大量资料中学习语言和知识。当前提示词、对话历史或额外提供的资料，也能影响这次回答。

使用时要注意信息从哪里来：

- **训练资料有时间范围**。例如训练资料截止到2022年，模型可能已经知道巴黎将举办2024年奥运会，因为主办城市早已公布；但赛后才产生的某场比赛结果，不能只靠这批训练资料得知。是否能回答，还取决于后来是否更新过模型、提供过资料，或接入了检索工具。
- **本地信息需要提供**。学校的最新开放时间、本节课安排、这台机器人昨天的实验记录，不能假定模型已经知道。

例如询问“我们实训室今天几点开放”，需要有本校的有效安排。只在提示词里写“你知道开放时间”，不会自动取得这份资料。


### 2.3 流畅的回答也可能有错

模型在资料不足或判断错误时，仍可能生成语气肯定、细节完整的回答。这类缺乏依据或与事实不符的生成内容，常被称为**幻觉**。

例如题目把人物或事件混在一起，模型可能顺着错误前提编下去，也可能指出错误。不同模型和提问方式的表现会有差别，不能要求某个旧例子每次都能复现。

检查回答时，要将人名、数字、时间等与可靠资料核对。核对可以由人完成，也可以借助检索和专门工具；不能只看回答是否通顺、有无自信。


### 2.4 BoosterOS SDK 中的接口

在 `BoosterOS SDK` 的 `Speech` 中有一个 `chat()` 接口，调用它可以启动语音对话任务。用法如下：

- `config`：`ChatConfig` 类型或 `None` 类型，语音对话配置；为 `None` 时使用默认配置；
- `**kwargs`：可选配置参数。

`chat()` 返回 `TaskHandle[None]`，代表整个持续会话。下面保留默认角色提示词，将音量设为0.6后体验。


**STEP 01：** 连接机器人

In [ ]:
from boosteros.robots.booster import BoosterRobot
from boosteros.brain import Speech
from boosteros.brain.speech import ChatConfig

robot = BoosterRobot()
speech = Speech(robot)

print("已连接：", robot.robot_info.model, robot.robot_info.serial_number)
print("当前音量：", robot.audio_manager.get_system_volume())

**STEP 02：** 初体验：开始一段问答

1. 向机器人问一个简单问题。
2. 说“停一下”“先别说”或“stop”，尝试打断它当前的回答。
3. 再问一个问题，观察会话是否继续。

打断当前回答与退出整个会话不同。体验结束时，按 Ctrl+C 或 Notebook 的中断按钮，让代码进入取消流程。`wait()` 等的是整个会话结束，用户问完一句、机器人答完一句都不代表会话已结束。

下一格只在上一项语音任务已结束时新建任务。取消后等待最多5秒，并查看 `done()`。若仍未结束，保留 `task`，运行 `task.cancel()`、`task.wait(timeout=5.0)` 继续检查，暂不开始下一项任务。


In [ ]:
if "task" in globals() and not task.done():
    print("上一项语音任务尚未结束，状态：", task.status)
    print("先运行 task.cancel()，再运行 task.wait(timeout=5.0)，确认 task.done() 为 True 后重试。")
else:
    task = speech.chat(
        config=ChatConfig(volume=0.6),
        enable_subtitle_log=True,
    )
    try:
        print("语音对话已启动，按 Ctrl+C 或 Notebook 的中断按钮结束本次会话。")
        task.wait()  # 等待整个会话结束，不是只等机器人回答一句话
    except KeyboardInterrupt:
        accepted = task.cancel()
        print("取消请求已提交：", accepted)
        task.wait(timeout=5.0)
    print("当前状态：", task.status, "；是否已结束：", task.done())
    if not task.done():
        print("任务尚未结束，保留 task 并继续检查；暂不启动下一段对话。")
    elif task.status == "FAILED":
        print("会话失败：", task.error)


## 三、配置聊天参数

可以通过音色、角色提示词和欢迎语，让对话更符合使用场景。


### 3.1 对话配置

在STEP 02中，我们初步配置了 `chat()` 接口的 `config` ——配置音量。该参数有如下字段可以配置：

ChatConfig 字段说明：

- `volume`字段，`float | None` 类型，默认值：`None`，代表：对话播放音量，范围 `0.0` 到 `1.0`；为 `None` 时不修改系统音量;
- `voice` 字段，`str` 类型，默认值："default"，代表：回复音色 ID，可通过 list_voices() 获取可用音色;
- `welcome_msg` 字段， `str | None` 类型，默认值：`None`，代表：对话开始时的欢迎语；为 `None` 时不设置欢迎语;
- `system_prompt`，`str | None` 类型，默认值：`None`，代表：角色提示词；为 None 时使用默认提示词。


**STEP 03：** 设置欢迎语和角色提示词


In [ ]:
if "task" in globals() and not task.done():
    print("上一项语音任务尚未结束，状态：", task.status)
    print("先运行 task.cancel()，再运行 task.wait(timeout=5.0)，确认 task.done() 为 True 后重试。")
else:
    task = speech.chat(
        config=ChatConfig(
            system_prompt="你是实训室讲解员。已知资料：本实训室使用K1机器人练习视觉检测、运动控制和语音交互。只根据这些资料回答，其他信息请说明暂不清楚。每次回答不超过两句。",
            welcome_msg="你好，欢迎来到实训室，你想了解哪项练习？",
            volume=0.5,
        ),
        enable_subtitle_log=True,
    )
    try:
        print("语音对话已启动，按 Ctrl+C 或 Notebook 的中断按钮结束本次会话。")
        task.wait()  # 等待整个会话结束，不是只等机器人回答一句话
    except KeyboardInterrupt:
        accepted = task.cancel()
        print("取消请求已提交：", accepted)
        task.wait(timeout=5.0)
    print("当前状态：", task.status, "；是否已结束：", task.done())
    if not task.done():
        print("任务尚未结束，保留 task 并继续检查；暂不启动下一段对话。")
    elif task.status == "FAILED":
        print("会话失败：", task.error)


`chat()` 接口的可选配置参数：

- 参数：`interrupt_speech_duration`，`int` 类型，默认值：`800`，代表：打断判定时长，单位 ms;
- 参数：`interrupt_keywords` ，`Sequence[str] | str` 类型，默认值：["停一下", "先别说", "stop"]，代表：触发打断的关键词;
- 参数：`enable_face_tracking` ， `bool` 类型，默认值：`False`，代表：是否启用对话过程中的人脸跟踪;
- 参数：`enable_subtitle_log`，`bool` 类型，默认值：`False`，代表：是否记录对话字幕日志。


当前回答通常受角色提示词、已有对话和新提问共同影响。角色提示词描述身份与要求，对话历史提供前面谈过的内容，新提问说明这次要回答什么。

上面的讲解员提示词还给了一小段实训室资料，便于回答有依据。提示词可以提供资料和要求，但不会凭空给机器人增加天气查询、数据库访问等工具。


### 3.2 同一模型的不同“人设”

保持模型不变，只换角色提示词，回答的语气、长度和侧重点就可能变化。这类角色设定常被称为**人设**。

提示词是在本次使用中提出要求，不会重新训练模型，也不保证模型每次都完全遵守。需要通过实际问答检查，再调整设定。


### 3.3 人设的四个部分

如何编写好用的人设呢？可以遵循下面四个角度进行人设编写：

- **身份**：你是谁（展厅迎宾员、足球解说员、科普老师、实训室讲解员）；
- **语气**：怎么说话（简短克制、激情、先说结论再举例）；
- **知识范围**：允许依据哪些资料回答（例如给出的展项资料）；
- **不能做什么**：不许做的事（不报价、不回答跟本展厅无关的问题）。

在编写这个内容时，大家常常只写「你是谁」，而不约定其他条件，这样机器人照样会聊到别的地方去。


### 3.4 “人设”写不好会出现什么

三种常见的问题：

- **写太短**：只写「你是一个迎宾员」。它就只能靠自己的理解去猜「迎宾员」是什么样，给模型发挥的自由度很高。
- **要求打架**：既要模型「回答不超过一句话」，又要「把展厅五个展项逐个介绍」。两条要求可能是相冲的，它只能舍一条，有时可能舍弃前面，有时可能舍弃后面。
- **管不住边界**：没写「不谈什么」。学生问「这个多少钱」，它就顺着答了，但是它没有相关信息的话，可能就开始编造内容了。

这三条都不是程序代码错误，程序照样能顺利执行。它们只在回答里表露出来——因此需要用具体问题测试，记录偏离要求的地方，再修改提示词。


## 四、实验：三种人设回答同一个问题

⚠️ **注意**：本课不让机器人走动，只用到麦克风与扬声器。机器人回答的时候不要同时说话，会互相干扰；音量先调好，别中途改。

三种人设、同一个问题、同一个音色——固定其他变量，观察不同人设的差别。

### 4.1 固定问题，测试语音识别效果

机器人接收的是现场说出的声音。语速、音量和环境噪声都会影响转写结果。

先用同一句问题试几次，练习稳定地说出问句。下面仍留6秒观察窗口，显示回调中最近一次文字；接口没有最终结果标志，这条文字只能作为本次观察记录。确认识别任务已结束后，再开始对话。


**STEP 04：** 多说几遍同一句话，看看识别效果

In [ ]:
import time

QUESTION = "请用一句话介绍一下机器人。"  # 后面几次都问同一句

if "task" in globals() and not task.done():
    print("上一项语音任务尚未结束，状态：", task.status)
    print("先运行 task.cancel()，再运行 task.wait(timeout=5.0)，确认 task.done() 为 True 后重试。")
else:
    last_text = ""

    def on_text(text):
        global last_text
        last_text = text
        print("文字回调：", text)

    task = speech.recognize_stream(on_result=on_text)
    print("请说出固定问题：", QUESTION)
    try:
        time.sleep(6)  # 留出6秒供观察文字回调，不代表识别一句话需要6秒
    finally:
        accepted = task.cancel()
        print("取消请求已提交：", accepted)
        task.wait(timeout=5.0)
    print("当前状态：", task.status, "；是否已结束：", task.done())
    if not task.done():
        print("任务尚未结束，保留 task，继续检查；暂不启动下一次识别或对话。")
    elif task.status == "FAILED":
        print("任务失败：", task.error)
    print("最近一次收到的文字（不保证为最终整句）：", last_text)


### 4.2 先问一次默认人设

不写 `system_prompt`，先看看它本来是什么样子。

**STEP 05：** 不写人设，问一次


In [ ]:
print("请对着机器人说出这句话：", QUESTION)

if "task" in globals() and not task.done():
    print("上一项语音任务尚未结束，状态：", task.status)
    print("先运行 task.cancel()，再运行 task.wait(timeout=5.0)，确认 task.done() 为 True 后重试。")
else:
    task = speech.chat(config=ChatConfig(volume=0.5), enable_subtitle_log=True)
    try:
        print("语音对话已启动，按 Ctrl+C 或 Notebook 的中断按钮结束本次会话。")
        task.wait()  # 等待整个会话结束，不是只等机器人回答一句话
    except KeyboardInterrupt:
        accepted = task.cancel()
        print("取消请求已提交：", accepted)
        task.wait(timeout=5.0)
    print("当前状态：", task.status, "；是否已结束：", task.done())
    if not task.done():
        print("任务尚未结束，保留 task 并继续检查；暂不启动下一段对话。")
    elif task.status == "FAILED":
        print("会话失败：", task.error)


### 4.3 三种人设各问一遍

三种身份各配一段设定，别的一律不动：同样的问句、同样的音色、同样的音量。

每次选择一种人设运行。体验后中断会话，确认 `task.done()` 为 `True`，再改 `name` 切换下一种人设；不按固定秒数猜测回答是否结束。

**STEP 06：** 三种人设各问一次


In [15]:
PROMPTS = {
    "解说员": "你是一名足球解说员，说话激情，语速快，喜欢用球场上的比喻。",
    "迎宾员": "你是一名展厅迎宾员，礼貌、克制，回答只允许一句话，回答完主动引导参观。",
    "科普老师": "你是一名给高职学生讲课的科普老师，先给结论，再举一个生活里的例子。",
}

name = "科普老师"
prompt = PROMPTS[name]

print(f"人设：{name}；请说出：{QUESTION}")

if "task" in globals() and not task.done():
    print("上一项语音任务尚未结束，状态：", task.status)
    print("先运行 task.cancel()，再运行 task.wait(timeout=5.0)，确认 task.done() 为 True 后重试。")
else:
    task = speech.chat(
        config=ChatConfig(system_prompt=prompt, volume=0.5),
        enable_subtitle_log=True,
    )
    try:
        print("语音对话已启动，按 Ctrl+C 或 Notebook 的中断按钮结束本次会话。")
        task.wait()  # 等待整个会话结束，不是只等机器人回答一句话
    except KeyboardInterrupt:
        accepted = task.cancel()
        print("取消请求已提交：", accepted)
        task.wait(timeout=5.0)
    print("当前状态：", task.status, "；是否已结束：", task.done())
    if not task.done():
        print("任务尚未结束，保留 task 并继续检查；暂不启动下一段对话。")
    elif task.status == "FAILED":
        print("会话失败：", task.error)


=== 人设：科普老师　请说出这句话：请用一句话介绍一下机器人。 ===
语音对话已启动，按 Ctrl+C 或程序停止运行按钮退出


[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 你好，有什么需要帮忙的吗？
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请用一
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请用一句话
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请用一句话介绍
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请用一句话介绍一下
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请用一句话介绍一下机器人
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请用一句话介绍一下机器人
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请用一句话介绍一下机器人。
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请用一句话介绍一下机器人。
[Booster OS] [boosteros.speech.booster] [INFO] Booster chat subtitle [user]: 请用一句话介绍一下机器人。
[Booster OS] [boosteros.speech.booster] [


正在停止语音对话...
=== 人设：科普老师对话结束，状态：CANCELLED


### 4.4 自己写一版

按身份、语气、知识范围、不能做什么四个部分，自己写一段。写完先读一遍，看看是否按照四个角度在编写人设提示词。

**STEP 07：** 换上自己写的人设


In [ ]:
MY_PROMPT = """你是展厅迎新讲解员，说话简短、语气友好。
已知展项：视觉检测演示、运动控制演示、语音交互演示。
只根据以上资料讲解，最多介绍三个展项，每项不超过一句话。
资料中没有的内容请说明暂不清楚；涉及报价或活动时间，请转给现场工作人员。"""

print("请对着机器人问同一个问题：", QUESTION)

if "task" in globals() and not task.done():
    print("上一项语音任务尚未结束，状态：", task.status)
    print("先运行 task.cancel()，再运行 task.wait(timeout=5.0)，确认 task.done() 为 True 后重试。")
else:
    task = speech.chat(
        config=ChatConfig(system_prompt=MY_PROMPT, volume=0.5),
        enable_subtitle_log=True,
    )
    try:
        print("语音对话已启动，按 Ctrl+C 或 Notebook 的中断按钮结束本次会话。")
        task.wait()  # 等待整个会话结束，不是只等机器人回答一句话
    except KeyboardInterrupt:
        accepted = task.cancel()
        print("取消请求已提交：", accepted)
        task.wait(timeout=5.0)
    print("当前状态：", task.status, "；是否已结束：", task.done())
    if not task.done():
        print("任务尚未结束，保留 task 并继续检查；暂不启动下一段对话。")
    elif task.status == "FAILED":
        print("会话失败：", task.error)


## 五、观察与记录

### 5.1 人设对照

| 人设 | 语气 | 回答长短 | 有没有聊到别的地方 | 是否满意 |
| :--- | :--- | :--- | :--- | :---: |
| 默认（不写设定） | 　　　　 | 　　　　 | 　　　　 | 　　　 |
| 解说员 | 　　　　 | 　　　　 | 　　　　 | 　　　 |
| 迎宾员 | 　　　　 | 　　　　 | 　　　　 | 　　　 |
| 科普老师 | 　　　　 | 　　　　 | 　　　　 | 　　　 |
| 自己写的那一版 | 　　　　 | 　　　　 | 　　　　 | 　　　 |

### 5.2 自己的人设里，四个角度各写了什么

| 部分 | 我写的内容 |
| :--- | :--- |
| 身份 | 　　　　　　　　　　　　　　　　 |
| 语气 | 　　　　　　　　　　　　　　　　 |
| 知识范围 | 　　　　　　　　　　　　　　　　 |
| 不能做什么 | 　　　　　　　　　　　　　　　　 |

### 5.3 检查点

- 是否能说清模型和设定分别管什么。
- 是否能指出哪一次回答跑出了设定的范围。
- 是否知道人设写得含糊会带来什么后果，并能根据观察修改。
- 是否确认前一项语音任务已结束，再切换人设；是否能区分打断一句回答与结束整个会话。

### 5.4 思考题

- 同一个问题问两次，回答不完全一样。这样的系统要进考试场景，还缺什么？
- 「不能做什么」这一条，为什么比「你是谁」更需要写？


## 六、扩展阅读：怎样减少缺乏依据的回答

模型生成回答时，流畅程度和事实准确性可能不一致。训练资料不足、提问带有错误前提、当前资料缺失等，都可能影响结果。

在实训室讲解这类任务中，可以这样处理：

- **提供有效资料**：给出本校实训室或展项说明，要求回答与资料相符。
- **约定回答范围**：只谈已给资料覆盖的内容，报价、开放时间等交由工作人员确认。
- **核对事实**：对人名、数字、日期等查阅原始资料；可由人完成，也可由程序调用专门工具辅助。
- **允许回答不知道**：资料没有说明时明确告知用户，而不是补出一个看似完整的答案。

这些方法可以降低错误风险，但提示词、检索工具和核查程序本身也有局限。模型能力、资料质量和使用流程都会影响结果。课堂上应把实际回答与已给资料对照，记录它是否遵守要求。
